# pipelineB.py - For CatBoost

In [5]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
import joblib

# LOAD THE RAW DATA

In [6]:
print("Loading raw data...")
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')

Loading raw data...


In [7]:
y_train = train['demand']
test_index = test['Index']

Drop Index and Demand, but KEEP geohash!

In [8]:
X_train_raw = train.drop(columns=['demand', 'Index'])
X_test_raw = test.drop(columns=['Index'])

In [9]:
print("--- PIPELINE B: Extracting time features ---")
def extract_time(df):
    df = df.copy()
    df[['hour', 'minute']] = df['timestamp'].str.split(':', expand=True).astype(float)
    return df.drop(columns=['timestamp'])

X_train = extract_time(X_train_raw)
X_test = extract_time(X_test_raw)

--- PIPELINE B: Extracting time features ---


In [10]:
print("--- PIPELINE B: Building and Running Pipeline ---")
numeric_features = ['Temperature', 'NumberofLanes', 'day', 'hour', 'minute']
categorical_features = ['geohash', 'RoadType', 'LargeVehicles', 'Landmarks', 'Weather']

--- PIPELINE B: Building and Running Pipeline ---


In [11]:
num_transformer = SimpleImputer(strategy='median')
cat_transformer = SimpleImputer(strategy='constant', fill_value='Missing')

In [12]:
pipeline_B = ColumnTransformer(transformers=[
    ('num', num_transformer, numeric_features),
    ('cat', cat_transformer, categorical_features)
])

Process the data

In [13]:
X_train_catboost = pipeline_B.fit_transform(X_train)
X_test_catboost = pipeline_B.transform(X_test)

Convert back to DataFrame

In [14]:
all_columns = numeric_features + categorical_features

df_train = pd.DataFrame(X_train_catboost, columns=all_columns)
df_train['demand'] = y_train.values

df_test = pd.DataFrame(X_test_catboost, columns=all_columns)
df_test['Index'] = test_index.values

In [15]:
print("--- PIPELINE B: Saving outputs ---")
df_train.to_csv('cleanedB/Train_CatBoost.csv', index=False)
df_test.to_csv('cleanedB/Test_CatBoost.csv', index=False)
joblib.dump(pipeline_B, 'Pipeline_B_Preprocessor.pkl')

--- PIPELINE B: Saving outputs ---


['Pipeline_B_Preprocessor.pkl']